In [2]:
from backtesting import Backtest, Strategy
import yfinance as yf
import pandas as pd

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
# descarga de datos
df_apple = yf.download('AAPL', period='3mo', interval='1d')

# limpieza
df_apple.columns = df_apple.columns.get_level_values(0)
df_apple.dropna(inplace=True)

[*********************100%***********************]  1 of 1 completed


In [4]:
# columna Returns
df_apple['Return'] = df_apple['Close'].pct_change() * 100
df_apple.dropna(inplace=True)

In [5]:
# crear classe que herede backtesting.Strategy
# se tienen que implementar 2 metodos init() y next()
class PercentilesStrategy(Strategy):
    # definir sl y tp
    sl_pct = 0.01
    tp_pct = 0.03
    window = 30

# aqui se calculan indicadores tecnicos, señales
# de las que depende la estrategia
# usando self.I(funcion, argumentos)
    def init(self):
        close = self.data.Close

        # es obligatorio usar self.I para hacer compatible el indicador con next()
        self.rolling_p25 = self.I(lambda x: pd.Series(x).rolling(self.window).quantile(0.25), close)
        self.rolling_p75 = self.I(lambda x: pd.Series(x).rolling(self.window).quantile(0.75), close)

    def next(self):
        price = self.data.Close[-1]

        if self.position:
            return
        
        if price < self.rolling_p25[-1]:
            self.open_long(price)

        elif price > self.rolling_p75[-1]:
            self.open_short(price)

    def open_long(self, price):
        self.buy(
            sl=price * (1 - self.sl_pct),
            tp=price * (1 + self.tp_pct)
        )

    def open_short(self, price):
        self.sell(
            sl=price * (1 + self.sl_pct),
            tp=price * (1 - self.tp_pct)
        )

In [6]:
# configuracion del backtest
bt = Backtest(df_apple, PercentilesStrategy, cash=10000, commission=0.002)

# ejecucion
stats = bt.optimize(
    sl_pct=[0.005, 0.01, 0.02],
    tp_pct=[0.02, 0.03, 0.05],
    window=range(10, 50, 5),
    maximize='Sharpe Ratio'
)

print(stats)

/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/pulpo/miniconda3/envs/trading_env/lib/python3.11/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. 

Start                     2026-01-27 00:00:00
End                       2026-04-23 00:00:00
Duration                     86 days 00:00:00
Exposure Time [%]                    77.04918
Equity Final [$]                   10094.4289
Equity Peak [$]                   10684.68046
Commissions [$]                     324.03634
Return [%]                            0.94429
Buy & Hold Return [%]                -0.43333
Return (Ann.) [%]                     3.95906
Volatility (Ann.) [%]                21.51384
CAGR [%]                              2.79228
Sharpe Ratio                          0.18402
Sortino Ratio                           0.276
Calmar Ratio                          0.69605
Alpha [%]                             1.00523
Beta                                  0.14064
Max. Drawdown [%]                    -5.68789
Avg. Drawdown [%]                    -2.78745
Max. Drawdown Duration       36 days 00:00:00
Avg. Drawdown Duration       11 days 00:00:00
# Trades                          

In [9]:
# grafico
bt.plot()

GridPlot(id='p2008', ...)